#### Stream processing (simulation)

- Uses Dataframe and Dataset APIs
- Micro-Batch Processing
- Exactly once guarantees
- Built-in Fault tolerance
- Scalable to large scale workloads
- Support various sources and sinks

##### Spark Structured Streaming

In [0]:
%sql
-- Only 20 records
SELECT * FROM json.`/Volumes/medallion_catalog/db_landing/vol_medallion/customer_stream/*` 

##### readStream

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

customers_schema = StructType(fields=[StructField('customer_id', IntegerType()),
                                      StructField('customer_name', StringType()),
                                      StructField('date_of_birth', DateType()),
                                      StructField('email', StringType()),
                                      StructField('member_since', DateType()),
                                      StructField('created_timestamp', TimestampType())
                                      ]
                              )



In [0]:
customers_df = (
    spark.readStream
    .format("json")
    .schema(customers_schema)
    .load("/Volumes/medallion_catalog/db_landing/vol_medallion/customer_stream/")
)



##### New columns:: file_path, ingestion_date

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_transformed_df = (
    customers_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

##### writeStream


In [0]:
streaming_query = (
    customers_transformed_df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/medallion_catalog/db_landing/vol_medallion/customer_stream/_checkpoint_stream")
        .trigger(availableNow=True)
        .toTable("medallion_catalog.db_bronze.customer_stream")
)

Second, third and fourth .json file manually dropped in the volume

In [0]:
%sql
-- 35 record after trigger(AvailableNow=True)
SELECT * FROM medallion_catalog.db_bronze.customer_stream

In [0]:
# streaming_query.stop()
# also terminate the cluster is a good idea

#### Spark Structured Streaming (append/update/complete)

Source Google AI overview

Summary of Performance: The "Update" output mode is designed to output only the rows that have changed since the last trigger. Its speed depends heavily on how efficiently Spark can compute and identify these changes, as well as the chosen execution mode.

- Micro-batch Mode: Provides robust, exactly-once guarantees with latencies in the order of hundreds of milliseconds per trigger.
- Real-Time Mode: Offers ultra-low latency (~1 ms) for suitable queries, though it has different fault-tolerance guarantees (at-least-once). 

In [0]:
streaming_query = (
    customers_transformed_df.writeStream
        .format("delta")
        # .outputMode("update")
        .option("checkpointLocation", "/Volumes/medallion_catalog/db_landing/vol_medallion/customer_stream/_checkpoint_stream")
        .trigger(availableNow=True)
        .toTable("medallion_catalog.db_bronze.customer_stream")
)